# ResNet18: 현환님 전처리 / +마스킹 / +마스킹+형상 v2 재활용
## Goal
**학습 전 노트북. 성능 수치는 아직 없음.** 로컬에서는 마스크·목록·분할을 준비했다.

| 조건 | 학습 후보 | 입력 |
|---|---:|---|
| v1 | 현환님 전처리 통과 2,652장 | 원본 고정 크롭 |
| v1_5 | 동일한 2,652장 | 현환님 베드 지도 마스킹 + 승인한 직선 경계 보정 |
| v2 | 통과+재활용 3,395장 | 통과는 v1_5와 동일, 추가 보류 사진만 형상 v2 마스킹 |

**②의 마스크에는 질감 v2도 형상 v2도 사용하지 않는다.** 현환님 원본 `image_metrics()`의
HSV/밝기 기반 베드 판별 지도를 사용한다. 밝기 V<70, 또는 H=10~60°, S=50~220,
V=40~180인 화소가 베드 후보다(이 S/V는 0~255 척도).
기존 마스크 보조 코드의 후처리(25×25 닫힘 연산으로 반짝이 틈 메우기,
0.5% 미만 작은 연결 후보 제외)를 적용해 베드/카메라를 가리고 출력물은 남긴다.
이후 **사용자가 100장으로 확인한 직선 보정**을 그대로 적용한다.
가장 큰 유지 영역 경계에서 화면 테두리를 제외하고 강건 직선을 적합한다.
경계점의 70% 이상이 직선의 12.5픽셀 이내이고 경계 길이가 300픽셀 이상일 때만
해당 직선의 출력물 쪽 반평면을 유지한다. 그 외에는 기존 HH 마스크 그대로다.
별도의 반짝이 섬 보정 코드나 새로운 기준값은 추가하지 않았다.
**이 후처리는 현환님 원본 선별기의 일부가 아니라 마스킹 추가 구현**이다.

**③은 ②를 유지하고, 기존 보류 중 형상 v2 SUPPORTED 804장 가운데 남는 출력물 면적 ≥ 0.1인 743장을 추가**한다.
면적 < 0.1인 61장(정상 33, 언더필 28)은 `rescue_small`로 따로 두고 진단 평가만 한다. 기준 0.1은 질감 게이트 통과 기준과 같고,
면적 5% 미만 12장 중 10장이 정상이라 "거의 회색 → 정상" 지름길을 막으려는 것이다(`RESCUE_MIN_AREA = 0`이면 804장 전부).
형상 v2는 내부적으로 질감 초기 후보를 쓰지만 이는 보류 재활용만의 기존 동작이다.
저장 마스크는 그대로 재사용한다. 통과 목록·재활용 목록에 새 선별 기준은 넣지 않았다.
마스크가 정확한 정답이라는 뜻은 아니며 베드 반사 누락/결함 틈 오제거를 확인해야 한다.

## Setup
1. 이 ipynb를 Colab의 **파일 → 노트북 업로드**로 연다.
2. **런타임 → 런타임 유형 변경 → GPU**를 선택한다.
3. Drive `산공종설` 폴더에 아래 두 ZIP을 둔다. 원본 ZIP은 기존 것을 그대로 사용한다.
   - `underfill_originals_4000.zip`
   - `resnet18_hh_straight_shape_rescue_bundle_20261005_v2.zip` (**새로 올릴 것**, 첫 판 묶음 대신)
4. 위에서부터 셀을 실행한다. 계정 비밀번호를 AI에게 주지 않는다.
   `drive.mount` 셀에서 본인이 Drive 접근을 승인한다.

**걸리는 시간 [추정]:** 입력 캐시 만들기 10~15분, 학습 15회(3조건×5fold)는 T4 기준 약 1~1.5시간.

Drive에는 원본·묶음 외에 결과/체크포인트 저장용 약 3GB 여유를 권장한다.
학습 셀이 ImageNet 사전학습 가중치를 처음 한 번 내려받는다(패키지 설치는 아님).

원본 ZIP에는 `normal/*.jpg`, `underfill_50/*.jpg`가 있어야 한다.
패키지 자동 설치는 하지 않는다. Colab 기본 PyTorch/torchvision을 확인하고 없으면 중단한다.


In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_v2.zip'
RUN_NAME = 'resnet18_hh_straight_shape_rescue_20261005_v2_run01'
IMG_SIZE = 512       # 원본 크롭 1000x1000과 모델 입력 해상도는 다름
BATCH = 16          # GPU 메모리가 부족하면 8로 바꾸고 RUN_NAME도 변경
ROUNDS = 10         # 고정 업데이트 예산: 아래 설명 참고
SEEDS = [0]         # 우선 예비 비교 1개 시드, 최종 근거에는 한계 기록
FOLDS = [0, 1, 2, 3, 4]
RESCUE_MIN_AREA = 0.1  # 재활용 사진의 최소 출력물 면적. 0이면 804장 전부
RUN_TRAINING = True  # 입력 확인만 하려면 False


### 1. GPU 확인 및 Drive 연결
본 학습은 Colab GPU에서만 실행한다. 결과와 체크포인트는 Drive에 저장한다.
런타임이 초기화되면 위에서부터 다시 실행한다. 환경/설정이 같을 때 완료한 fold는 건너뛰고
미완료 fold는 마지막 저장 round 다음부터 이어간다. round 도중 끊기면 그 round만 다시 돈다.


In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꿔줘.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)


### 2. ZIP을 로컬 SSD로 풀고 파일 무결성 확인
학습 중 JPG를 Drive에서 매번 읽지 않는다. 원본과 마스크를 `/content`로 복사한다.
압축을 푼 뒤 목록·해시·기존 5-fold/30프레임 buffer를 확인한다.


In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
from IPython.display import display
LOCAL = Path('/content/hh_straight_shape_rescue')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE = LOCAL / 'bundle'
ORIGINALS = LOCAL / 'originals'
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    # 원본은 크므로 같은 세션에서 재실행 시 파일 크기가 같은 복사본을 재사용
    if source.name == BUNDLE_ZIP_NAME or not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as z:
        for member in z.infolist():
            resolved = (target / member.filename).resolve()
            assert resolved.is_relative_to(target.resolve()), 'ZIP 경로 범위 오류'
        z.extractall(target)
    print('압축 해제:', source.name)

extract_zip(DRIVE_DIR / BUNDLE_ZIP_NAME, BUNDLE)
extract_zip(DRIVE_DIR / ORIGINAL_ZIP_NAME, ORIGINALS)
hashes = json.loads((BUNDLE / 'checksums.json').read_text())
for name, expected in hashes.items():
    assert hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == expected, name
subprocess.run([sys.executable, str(BUNDLE / 'train_resnet18.py'),
                '--bundle', str(BUNDLE), '--verify-only'], check=True)
print('묶음 파일 해시 검증 완료:', len(hashes))


## Data & Methods
- 원본 normal 2,000장 / underfill_50 2,000장. 라벨은 출력 조건이며 각 프레임의 결함 가시성을 보장하지 않는다.
- 고정 ROI: 원본 좌표 x=250~1250, y=0~1000. 1000×1000 캔버스에서 확대 없이 마스킹한 뒤
  모델 입력 512×512로 bilinear antialias 리사이즈한다. 제외 영역 RGB=(124,116,104).
  검토용 빨간 표시 이미지는 학습에 쓰지 않는다.
- 같은 기존 시간 블록 5-fold, 경계 30프레임 buffer. 세 조건의 평가 사진은 같다.
  통과(primary)와 재활용(rescue)을 별도로 채점한다.
- v1/v1_5에서 재활용 사진의 점수는 **강제로 입력한 진단 점수**다.
  기존 게이트가 실제로 판정한 것으로 간주하지 않는다. 보류=정상도 아니다.
- ImageNet ResNet18 전이학습/전체 미세조정. 초기 가중치, 정규화, 증강, 옵티마이저는 동일.
  train에만 좌우 반전 p=.5, 밝기/대비 ±.1. AdamW backbone 1e-4, head 1e-3, wd 1e-4.
- **round=고정 업데이트 예산 단위**: 해당 fold의 v1 학습 수/batch를 올림한 step 수.
  모든 조건이 같은 round 수를 사용한다. v2는 사진이 더 많아 사진당 평균 반복 수가 더 적다.
  고정 마지막 round 평가이며 조기 종료/최고 검증점수 모델 선택은 하지 않는다.
  기존 `train_pilot_classifier.py`(20 epoch) 결과와는 학습량이 달라 수치를 직접 비교하지 않는다. 이 노트북 안의 세 조건끼리만 비교한다.
- 실시간 관점 지표: `holds.csv`(보류 수, 최장 연속 보류, 4장/초 가정이면 ÷4초), `vote.csv`(판정 사진만 촬영 순서로 5장 다수결).
- 기본 3조건×5fold×1seed=15회. seed 1개는 예비 비교다. AMP와 빠른 cuDNN 모드 때문에
  완전한 비트 단위 재현성은 보장하지 않는다.

### Key Assumptions / 한계
클래스별 출력 1회이므로 새 출력 실험 일반화 검증이 아니다. 형상 마스크는 오프라인 공정표와
촬영 문맥을 사용했으며 신규 출력 실시간 자동 동기화는 검증되지 않았다.
통과 사진 마스크는 100장 사용자 육안 검토를 거쳤고, 기존 1,348장 형상 v2 육안 검토와 구분해야 한다.
100장 검토 결과와 최종 마스크의 픽셀 일치를 검사한다. 정답 마스크 IoU 검증은 없음.
회색 면적/촬영 시점/조명 지름길은 해결됐다고 주장하지 않는다.
804장 중 61장은 고정 ROI 안 남는 면적이 0.1 미만이라 v2 학습·판정에서 뺐다(진단 평가만).


In [ ]:
import pandas as pd
manifest = pd.read_csv(BUNDLE / 'manifest.csv')
prep = json.loads((BUNDLE / 'preparation.json').read_text(encoding='utf-8'))
display(pd.read_csv(BUNDLE / 'coverage.csv'))
print('통과 사진 베드 마스크 생성:', prep['primary_mask_generated'])
print('직선 보정/기존 마스크 유지:', prep['primary_boundary_status_counts'])
print('승인한 100장과 픽셀 일치:', prep['approved_preview_equal_pixels'])
print('통과에 질감 v2 사용:', prep['primary_texture_v2_used'],
      '/ 통과에 형상 v2 사용:', prep['primary_shape_v2_used'])
selected = manifest[manifest.cohort != 'held'].reset_index(drop=True)
assert len(selected) == 3456
print('선택 사진 중 마스크 면적 1% 미만:', int((selected.mask_area < .01).sum()))
display(selected.groupby(['cohort', 'label']).mask_area.agg(['count', 'min', 'median', 'max']))
display(selected[['source','cohort','mask_status','mask_area']].head(8))
small = selected[(selected.cohort == 'rescue') & (selected.mask_area < RESCUE_MIN_AREA)]
print(f'v2 학습에서 뺄 재활용(면적<{RESCUE_MIN_AREA}):', len(small), small.label.value_counts().to_dict())


### 3. 실제 학습 입력 생성 및 확인
원본·마스킹 입력 두 벌을 메모리맵으로 저장한다(512일 때 약 5.1 GiB, 로컬 SSD).
영역 내부 픽셀이 유지되는지, 모든 입력 사진·마스크의 SHA256이 맞는지 확인한다.
아래 미리보기에서 **원본과 마스킹 결과를 직접 보고** 학습 셀로 넘어간다.


In [ ]:
def logged_run(command, log_path):
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', env=env)
        for line in process.stdout:
            log.write(line); log.flush()
            # 다운로드/긴 로그는 파일로; 주요 진행만 노트북에 출력
            if any(word in line for word in ('입력 ', '완료', 'round ', 'Traceback', 'Error', '이미 완료', '보류 ', '재활용 면적')):
                print(line.rstrip(), flush=True)
        result = process.wait()
    assert result == 0, f'실행 실패: {log_path} 확인'

common = [sys.executable, str(BUNDLE / 'train_resnet18.py'),
          '--bundle', str(BUNDLE), '--originals', str(ORIGINALS),
          '--cache', str(CACHE), '--output', str(OUTPUT),
          '--size', str(IMG_SIZE), '--batch', str(BATCH), '--rounds', str(ROUNDS),
          '--rescue-min-area', str(RESCUE_MIN_AREA)]
logged_run(common + ['--prepare-only'], OUTPUT / 'input_preparation.log')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
raw = np.load(CACHE / 'raw.npy', mmap_mode='r')
masked = np.load(CACHE / 'masked.npy', mmap_mode='r')
sample = []
for cohort in ('primary', 'rescue'):
    for label in ('normal', 'underfill_50'):
        sample.append(int(selected[(selected.cohort == cohort) & (selected.label == label)].index[0]))
for frame in (2127, 3654):
    found = selected.index[selected.frame_index == frame].tolist()
    if found: sample.append(found[0])
fig, axes = plt.subplots(len(sample), 2, figsize=(9, 3.5 * len(sample)), squeeze=False)
for row, index in enumerate(sample):
    item = selected.iloc[index]
    method = 'HH + approved straight boundary' if item.cohort == 'primary' else 'Shape v2 rescue'
    for col, (array, title) in enumerate(((raw, 'Raw fixed crop'), (masked, method + ' masked'))):
        axes[row,col].imshow(array[index].transpose(1,2,0))
        axes[row,col].axis('off')
        axes[row,col].set_title(f'{title}: {item.label} #{item.frame_index}\n{item.mask_status}, area={item.mask_area:.3%}', fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT / 'actual_input_preview.png', dpi=120, bbox_inches='tight')
plt.show()


### 4. 학습 실행
아래 셀이 실제 GPU 학습이다. 아직 결과가 없으므로 성능이 좋아질 것이라고 단정하지 않는다.
진행 및 체크포인트는 Drive에 저장한다. 설정을 바꿀 때는 RUN_NAME도 바꾼다.
중단한 실행은 같은 설정과 같은 환경에서 재개한다. 체크포인트는 본인 실행 파일만 사용한다.


In [ ]:
if RUN_TRAINING:
    command = common + ['--seeds'] + list(map(str, SEEDS)) + ['--folds'] + list(map(str, FOLDS))
    logged_run(command, OUTPUT / 'training.log')
else:
    print('입력 확인까지만 실행. 학습하지 않았어.')


## Results / Checks
완료 fold의 OOF(해당 사진을 학습하지 않은 모델 예측)만 집계한다.
`primary`와 `rescue` 성능을 먼저 비교한다. 세 조건의 `all_common`은 같은 사진이지만,
`pipeline_accepted`는 판정 범위가 달라 직접 순위를 매기지 않는다.
`rescue_small`(면적<0.1)은 어느 조건도 판정하지 않는 사진이라 진단용으로만 본다.
'클린 사진만 학습하면 주변부 성능이 낮아진다'는 가설은 **v1_5↔v2의 rescue 결과**와
paired_comparison의 동일 사진 변화로 검토한다. 차이가 없거나 반대여도 그대로 보고한다.


In [ ]:
if (OUTPUT / 'summary.csv').exists():
    summary = pd.read_csv(OUTPUT / 'summary.csv')
    columns = ['version','seed','cohort','completed_folds','all_folds_complete',
               'n','accuracy','balanced_accuracy','auroc','fp','fn']
    display(summary[columns].round(4))
    display(pd.read_csv(OUTPUT / 'paired_comparison.csv').round(4))
    print('[보류와 최장 연속 보류]'); display(pd.read_csv(OUTPUT / 'holds.csv').round(4))
    print('[5장 다수결]'); display(pd.read_csv(OUTPUT / 'vote.csv').round(4))
    plot_data = summary[summary.cohort.isin(['primary','rescue'])]
    if plot_data.all_folds_complete.all():
        for seed in sorted(plot_data.seed.unique()):
            table = plot_data[plot_data.seed == seed].pivot(index='version', columns='cohort', values='balanced_accuracy')
            ax = table.reindex(['v1','v1_5','v2']).plot.bar(figsize=(8,4), color=['#2463a2','#e39c24'], rot=0)
            ax.set_ylim(0,1); ax.set_ylabel('Balanced accuracy (OOF)')
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.set_title(f'Same validation photos: primary n=2652 / rescue n=743, seed {seed}')
            for index, container in enumerate(ax.containers):
                if index == 1:
                    for bar in container: bar.set_hatch('//')
                ax.bar_label(container, labels=[f'{bar.get_height():.1%}' for bar in container], padding=3, fontsize=9)
            ax.grid(axis='y', alpha=.2); ax.figure.tight_layout()
            ax.figure.savefig(OUTPUT / f'oof_balanced_accuracy_seed{seed}.png', dpi=150)
            plt.show()
    else:
        print('일부 fold만 완료됨. 최종 비교로 해석하지 말 것.')
else:
    print('완료한 학습 결과 없음. 실제 학습 후 이 셀을 다시 실행해줘.')


## Next Steps
Drive 결과를 확인한 다음 논문/팀 비교에 쓸 CSV·설정·로그를 아래 ZIP으로 내려받는다.
큰 모델 체크포인트는 ZIP에서 제외하며 Drive에 유지한다.
그래프는 실제 학습 후 생성되므로 이번 작성 단계에서는 확인하지 못했다.
Colab 전체 실행도 아직 하지 않았다. 오류가 나면 로그와 마지막 오류 셀을 공유한다.
결과 보관을 확인한 후 **런타임 → 연결 해제 및 삭제**로 GPU 런타임을 반납한다.


In [ ]:
result_zip = DRIVE_DIR / f'{RUN_NAME}_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.rglob('*'):
        if path.is_file() and path.suffix in ('.csv','.json','.log','.png'):
            z.write(path, path.relative_to(OUTPUT))
print('결과 위치:', OUTPUT)
print('공유용 결과 ZIP:', result_zip)


### 런타임 반납
결과 ZIP 위치를 확인한 뒤 실행한다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()